In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import time
import numpy as np
import os
import pandas as pd
from collections import defaultdict
from constants import WORD_TYPES, PROPER_NOUN_TYPES
pd.options.display.max_columns = 100
pd.options.display.max_rows = 130

In [35]:
from settings_shorts import (
    load_data_settings, audio_settings, load_video_configs
)
from utils_shorts import (
    load_category_data, load_examples_data, generate_and_zip_audio_files, stitch_audios
    , draw_vocab_list_whole_image, create_directories, create_video_with_highlights, create_video_without_highlights
)

# Get list of words for chosen hanzi

In [25]:
audio_settings = {
    'voice_name_zh': 'zh-CN-XiaoxiaoNeural',
    'audio_plan': 'ctitle_c2word',
    'pause_ms_beginning': 150,
    'pause_ms_within_word': 200,
    'pause_ms_between': 500,
}

In [36]:
def enhance_data_settings(data_settings):
    data_settings['output_path_base'] = 'output/shared_char_shorts/'
    data_settings['output_path'] = os.path.join(
        data_settings['output_path_base'],
        data_settings['shared_char']
    )
    data_settings['output_path_audio'] = os.path.join(
        data_settings['output_path'],
        'audio_files'
    )
    data_settings['output_path_images'] = os.path.join(
        data_settings['output_path'],
        'images'
    )
    return data_settings

data_settings = {
    'shared_char': '车',
    'char_pinyin': 'chē',
    'char_english': 'vehicle',
    'max_priority': 4,
    'min_adu': 3,
    'min_per': 3,
    'types_allowed': WORD_TYPES + PROPER_NOUN_TYPES,
    'sort_cols': ['priority', 'cat_v3', 'pinyin'],
    'sort_ascending': [True, True, True],
    'words_rmv': ['马里奥赛车', '停车标志'],
    'n_words_per_video': 12,
    'current_part': 1,
    'n_parts': 2,
    'current_part_index_range': (0, 11),
}
data_settings = enhance_data_settings(data_settings)
create_directories(data_settings)
data_settings

{'shared_char': '车',
 'char_pinyin': 'chē',
 'char_english': 'vehicle',
 'max_priority': 4,
 'min_adu': 3,
 'min_per': 3,
 'types_allowed': ['word',
  'prefix',
  'suffix',
  'abbreviation',
  'multi_word',
  'verb_ending',
  'proper noun'],
 'sort_cols': ['priority', 'cat_v3', 'pinyin'],
 'sort_ascending': [True, True, True],
 'words_rmv': ['马里奥赛车', '停车标志'],
 'n_words_per_video': 12,
 'current_part': 1,
 'n_parts': 2,
 'current_part_index_range': (0, 11),
 'output_path_base': 'output/shared_char_shorts/',
 'output_path': 'output/shared_char_shorts/车',
 'output_path_audio': 'output/shared_char_shorts/车/audio_files',
 'output_path_images': 'output/shared_char_shorts/车/images'}

In [ ]:
from utils_data import load_raw_data, check_dups
truly_load_data = False
if truly_load_data:
    df_all_vocab = load_raw_data()
    df_all_vocab.to_csv('static/latest_data.csv', index=False)
else:
    df_all_vocab = pd.read_csv('static/latest_data.csv')
    print('!!!!!!!! WARNING: not truly loading data !!!!!!!!')

df_dups = check_dups(df_all_vocab)
print(df_all_vocab.shape)
print(f'# duplicate vocab: {len(df_dups)}')
df_all_vocab.head(3)

(7488, 38)
# duplicate vocab: 0


,id,chinese,pinyin,english,type,priority,category1,category2,cat_v3,cat2_v3,cat3_v3,hsk_level,known,known_pinyin_prompt,known_english_prompt,quality,word1,word1_english,word2,word2_english,word3,word3_english,word4,word4_english,voice_zh,voice_en,video_notes,sentence,sentence_pinyin,sentence_english,date,source1,source2,funny,per,adu,slang,phonetic
0,1,房贷,fáng dài,mortgage,word,1.0,life,NaN,Finance & Economy,NaN,NaN,NONE,1.0,1.0,2.0,1.0,房子,house,贷款,loan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,他每个月都要还房贷,Ta měi gè yuè dōu yào huán fángdài,He has to pay his mortgage every month,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
1,2,白天,bái tiān,daytime,word,2.0,time,NaN,Time,NaN,NaN,1,2.0,1.0,1.0,1.0,白,white,天,day,NaN,NaN,NaN,NaN,NaN,NaN,NaN,白天很热晚上比较凉快,Báitiān hěn rè wǎnshàng bǐjiào liángkuai,It is hot in the daytime and cooler at night,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
2,3,组成,zǔ chéng,to form;make up,word,3.0,general,NaN,Language & Expression,NaN,NaN,2,5.0,5.0,5.0,3.0,组,set,成,become,NaN,NaN,NaN,NaN,NaN,NaN,NaN,水是由氢和氧组成的,Shuǐ shì yóu qīng hé yǎng zǔchéng de,Water is made up of hydrogen and oxygen,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN


In [23]:
def get_filtered_words(df_all_vocab, data_settings):
    df_filtered = df_all_vocab[
        (df_all_vocab['chinese'].str.contains(data_settings['shared_char'])) &
        (df_all_vocab['type'].isin(data_settings['types_allowed'])) &
        (df_all_vocab['priority'] <= data_settings['max_priority']) &
        (~(df_all_vocab['adu'] < data_settings['min_adu'])) &
        (~(df_all_vocab['per'] < data_settings['min_per'])) &
        (~(df_all_vocab['chinese'].isin(data_settings['words_rmv'])))
    ].sort_values(by=data_settings['sort_cols'], ascending=data_settings['sort_ascending']).reset_index(drop=True)
    return df_filtered

df_filt = get_filtered_words(df_all_vocab, data_settings)
print(['车'] + df_filt['chinese'].values.tolist())
print(len(df_filt))
df_filt.head()

['车', '出租车', '电动车', '堵车', '公共汽车', '火车', '汽车', '自行车', '救护车', '晕车', '车库', '车位', '车祸', '自动驾驶汽车', '车牌', '共享单车', '卡车', '缆车', '摩托车', '停车场', '网约车', '购物车', '三轮车', '婴儿车']
23


,id,chinese,pinyin,english,type,priority,category1,category2,cat_v3,cat2_v3,cat3_v3,hsk_level,known,known_pinyin_prompt,known_english_prompt,quality,word1,word1_english,word2,word2_english,word3,word3_english,word4,word4_english,voice_zh,voice_en,video_notes,sentence,sentence_pinyin,sentence_english,date,source1,source2,funny,per,adu,slang,phonetic
0,426,出租车,chū zū chē,taxi,word,1.0,travel,NaN,Transportation,NaN,NaN,2,1.0,1.0,1.0,2.0,出,to go out,租,rent,车,car,NaN,NaN,NaN,NaN,NaN,他打了一辆出租车,tā dǎ le yī liàng chū zū chē,he took a taxi,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
1,427,电动车,diàn dòng chē,electric vehicle,word,1.0,travel,NaN,Transportation,NaN,NaN,4,1.0,1.0,1.0,2.0,电,electric,动,move,车,vehicle,NaN,NaN,NaN,NaN,NaN,他买了一辆电动车,tā mǎi le yī liàng diàn dòng chē,he bought an electric scooter,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
2,1249,堵车,dǔ chē,traffic jam,word,1.0,travel,NaN,Transportation,NaN,NaN,4,4.0,1.0,1.0,2.0,堵塞,blockage,车,car,NaN,NaN,NaN,NaN,NaN,NaN,NaN,我们在高速公路上堵车了,Wǒmen zài gāosù gōnglù shàng dǔchē le,We got stuck in traffic on the highway,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
3,425,公共汽车,gōng gòng qì chē,bus,word,1.0,travel,NaN,Transportation,NaN,NaN,2,1.0,1.0,1.0,2.0,公共,public,汽车,car,NaN,NaN,NaN,NaN,NaN,NaN,NaN,我们坐公共汽车去市中心,wǒ men zuò gōng gòng qì chē qù shì zhōng xīn,we took a bus to the city center,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
4,395,火车,huǒ chē,train,word,1.0,travel,NaN,Transportation,NaN,NaN,1,1.0,1.0,1.0,2.0,火,fire,车,car,NaN,NaN,NaN,NaN,NaN,NaN,NaN,他们坐火车去北京,tā men zuò huǒ chē qù běi jīng,they took the train to Beijing,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN


In [45]:
def load_video_configs():
    BG_SIZE = (720, 1280)
    video_configs = {
        'bg_size': BG_SIZE,
        'bg_color': 'white',
        'text_color': 'black',

        'max_line_length': BG_SIZE[0] - 160,
        'decrease_font_step_size': 1,
        'font_path': '/System/Library/Fonts/STHeiti Medium.ttc',

        'highlight_rect_x_buffer': 24,
        'highlight_rect_color': [0, 255, 0],
        'highlight_rect_opacity': 0.5,

        'title_settings': {
            'x': 50,
            'y': 40,
            'spacing': 8,
            'align': 'center',
            'font_size': {'chinese': 48, 'pinyin': 32, 'english': 32},
            'fill': {'chinese': '#000000', 'pinyin': '#222222', 'english': '#222222'},
        },

        'words_settings': {
            'x': {'chinese': 50, 'pinyin': 185, 'english': 440},
            'max_line_length_buffer_size': {'chinese': 15, 'pinyin': 30, 'english': 60},
            'max_line_length': {},
            'y_gap': 50,
            'spacing': 40,
            'font_size': {'chinese': 32, 'pinyin': 32, 'english': 32},
            'align': {'chinese': 'left', 'pinyin': 'left', 'english': 'left'},
            'fill': {'chinese': '#000000', 'pinyin': '#000000', 'english': '#000000'},
        },

        'horizontal_line': {
            'y_gap': 20,
            'x': 10,
            'color': "#1E90FF",
            'width': 10,
        },

        'bottom_line': {
            'y_gap': 10,
            'x': 10,
            'color': "#1E90FF",
            'width': 10,
        },

        'logo': {
            'font_name': 'Arial Black',
            'font_size': 20,
            'x': BG_SIZE[0] - 50,
            'y': 1130,
            'color1': "#3E78D6",
            'color2': "#2FDDFC",
        },

        'category_index': {
            'index_value': 1,
            'index_total': 100,
            'font_name': 'Arial Black',
            'font_size': 48,
            'x': 240,
            'y': 1110,
            'color1': "#000000",
            'color2': "#777777",
        },
    }
    video_configs['horizontal_line']['y'] = video_configs['title_settings']['y'] + \
        video_configs['title_settings']['font_size']['chinese']+ \
        video_configs['title_settings']['font_size']['pinyin'] + \
        video_configs['title_settings']['font_size']['english'] + \
        video_configs['horizontal_line']['y_gap'] + \
        2*video_configs['title_settings']['spacing']

    video_configs['bottom_line']['y'] = video_configs['category_index']['y'] - \
        video_configs['bottom_line']['y_gap']

    video_configs['words_settings']['y'] = video_configs['horizontal_line']['y'] + \
        video_configs['words_settings']['y_gap']
    video_configs['words_settings']['max_line_length']['chinese'] = video_configs['words_settings']['x']['pinyin'] - video_configs['words_settings']['x']['chinese'] - video_configs['words_settings']['max_line_length_buffer_size']['chinese']
    video_configs['words_settings']['max_line_length']['pinyin'] = video_configs['words_settings']['x']['english'] - video_configs['words_settings']['x']['pinyin'] - video_configs['words_settings']['max_line_length_buffer_size']['pinyin']
    video_configs['words_settings']['max_line_length']['english'] = BG_SIZE[0] - video_configs['words_settings']['x']['english'] - video_configs['words_settings']['max_line_length_buffer_size']['english']
    return video_configs

video_configs = load_video_configs()

# Generate audio

In [31]:
vocab_word_list = df_filt[
    (df_filt.index >= data_settings['current_part_index_range'][0]) &
    (df_filt.index <= data_settings['current_part_index_range'][1])
    ]['chinese'].values.tolist()
print(len(vocab_word_list))
vocab_word_list

12


['出租车', '电动车', '堵车', '公共汽车', '火车', '汽车', '自行车', '救护车', '晕车', '车库', '车位', '车祸']

In [41]:
import pandas as pd
import numpy as np
import os
import time
import shutil
import datetime
from edge_tts import Communicate
from collections import defaultdict
from PIL import Image, ImageDraw, ImageFont
from moviepy import ImageClip, CompositeVideoClip, AudioFileClip, ColorClip, VideoFileClip
from utils_video import determine_if_text_size_too_big
from pydub import AudioSegment

def stitch_audios(audio_settings, data_settings, example_words):
    dict_audio_durations = defaultdict(list)
    if audio_settings['audio_plan'] == 'ctitle_c2word':
        current_start_time = 0

        # beginning pause
        pause_beginning = AudioSegment.silent(duration=audio_settings['pause_ms_beginning'])
        combined = pause_beginning
        dict_audio_durations['audio_path'].append('pause_beginning')
        dict_audio_durations['duration'].append(audio_settings['pause_ms_beginning'] / 1000)
        dict_audio_durations['start_time'].append(current_start_time)
        current_start_time += audio_settings['pause_ms_beginning'] / 1000
        dict_audio_durations['end_time'].append(current_start_time)

        # title
        title_audio_path = f"{data_settings['output_path_audio']}/{data_settings['shared_char']}.mp3"
        audio = AudioSegment.from_mp3(title_audio_path)
        combined += audio
        dict_audio_durations['audio_path'].append(title_audio_path)
        dict_audio_durations['duration'].append(audio.duration_seconds)
        dict_audio_durations['start_time'].append(current_start_time)
        current_start_time += audio.duration_seconds
        dict_audio_durations['end_time'].append(current_start_time)

        # words
        for _, word in enumerate(example_words):
            # inter-word pause
            pause_inter_word = AudioSegment.silent(duration=audio_settings['pause_ms_between'])
            combined += pause_inter_word
            dict_audio_durations['audio_path'].append('inter_word_pause')
            dict_audio_durations['duration'].append(audio_settings['pause_ms_between'] / 1000)
            dict_audio_durations['start_time'].append(current_start_time)
            current_start_time += audio_settings['pause_ms_between'] / 1000
            dict_audio_durations['end_time'].append(current_start_time)
            
            # word audio
            word_audio_path = f"{data_settings['output_path_audio']}/{word}.mp3"
            audio = AudioSegment.from_mp3(word_audio_path)
            combined += audio 
            dict_audio_durations['audio_path'].append(word_audio_path)
            dict_audio_durations['duration'].append(audio.duration_seconds)
            dict_audio_durations['start_time'].append(current_start_time)
            current_start_time += audio.duration_seconds
            dict_audio_durations['end_time'].append(current_start_time)

            # within-word pause
            pause_within_word = AudioSegment.silent(duration=audio_settings['pause_ms_within_word'])
            combined += pause_within_word
            dict_audio_durations['audio_path'].append('within_word_pause')
            dict_audio_durations['duration'].append(audio_settings['pause_ms_within_word'] / 1000)
            dict_audio_durations['start_time'].append(current_start_time)
            current_start_time += audio_settings['pause_ms_within_word'] / 1000
            dict_audio_durations['end_time'].append(current_start_time)

            # word again audio
            combined += audio 
            dict_audio_durations['audio_path'].append(word_audio_path)
            dict_audio_durations['duration'].append(audio.duration_seconds)
            dict_audio_durations['start_time'].append(current_start_time)
            current_start_time += audio.duration_seconds
            dict_audio_durations['end_time'].append(current_start_time)

    # export the combined audio file
    combined.export(f"{data_settings['output_path_audio']}/!combined_part{data_settings['current_part']}.mp3", format="mp3")
    print(f'Audio duration: {combined.duration_seconds:.1f}s')

    # Add in static slide audio into dataframe of audio durations
    df_durations = pd.DataFrame(dict_audio_durations)
    return df_durations

df_durations = stitch_audios(audio_settings, data_settings, vocab_word_list)
df_durations.head()

Audio duration: 42.5s


,audio_path,duration,start_time,end_time
0,pause_beginning,0.150,0.000,0.150
1,output/shared_char_shorts/车/audio_files/车.mp3,1.080,0.150,1.230
2,inter_word_pause,0.500,1.230,1.730
3,output/shared_char_shorts/车/audio_files/出租车.mp3,1.416,1.730,3.146
4,within_word_pause,0.200,3.146,3.346


# Genereate image

In [75]:
from utils_shorts import draw_logo, draw_resized_text_on_image

def draw_vocab_list_whole_image(video_configs, data_settings, df_filt):
    original_img = Image.new("RGB", video_configs['bg_size'], color=video_configs['bg_color'])
    draw = ImageDraw.Draw(original_img)
    draw_logo(draw, video_configs)

    title_text_settings = {}
    title_text_settings['chinese'] = {
        'text': data_settings['shared_char'],
        'font_path': video_configs['font_path'],
        'font_size': video_configs['title_settings']['font_size']['chinese'],
        'y': video_configs['title_settings']['y'],
        'spacing': video_configs['title_settings']['spacing'],
        'align': video_configs['title_settings']['align'],
        'fill': video_configs['title_settings']['fill']['chinese'],
        'max_line_length': video_configs['max_line_length'],
    }
    title_text_settings['pinyin'] = {
        'text': data_settings['char_pinyin'],
        'font_path': video_configs['font_path'],
        'font_size': video_configs['title_settings']['font_size']['pinyin'],
        'y': video_configs['title_settings']['y'] + video_configs['title_settings']['font_size']['chinese'] + video_configs['title_settings']['spacing'],
        'spacing': video_configs['title_settings']['spacing'],
        'align': video_configs['title_settings']['align'],
        'fill': video_configs['title_settings']['fill']['chinese'],
        'max_line_length': video_configs['max_line_length'],
    }
    title_text_settings['english'] = {
        'text': data_settings['char_english'],
        'font_path': video_configs['font_path'],
        'font_size': video_configs['title_settings']['font_size']['english'],
        'y': video_configs['title_settings']['y'] + video_configs['title_settings']['font_size']['chinese'] + video_configs['title_settings']['font_size']['pinyin'] + 2*video_configs['title_settings']['spacing'],
        'spacing': video_configs['title_settings']['spacing'],
        'align': video_configs['title_settings']['align'],
        'fill': video_configs['title_settings']['fill']['chinese'],
        'max_line_length': video_configs['max_line_length'],
    }
    draw_resized_text_on_image(draw, title_text_settings['chinese'], video_configs, is_centered=True)
    draw_resized_text_on_image(draw, title_text_settings['pinyin'], video_configs, is_centered=True)
    draw_resized_text_on_image(draw, title_text_settings['english'], video_configs, is_centered=True)

    # 4. Horizontal line
    draw.line([
        (video_configs['horizontal_line']['x'], video_configs['horizontal_line']['y']),
        (video_configs['bg_size'][0] - video_configs['horizontal_line']['x'], video_configs['horizontal_line']['y'])],
        fill=video_configs['horizontal_line']['color'],
        width=video_configs['horizontal_line']['width'],
        joint=None)
    draw.line([
        (video_configs['bottom_line']['x'], video_configs['bottom_line']['y']),
        (video_configs['bg_size'][0] - video_configs['bottom_line']['x'], video_configs['bottom_line']['y'])],
        fill=video_configs['bottom_line']['color'],
        width=video_configs['bottom_line']['width'],
        joint=None)
    
    # Write part number, if applicable
    if 'current_part' in data_settings:
        part_text_settings = {
            'text': f"Part\n{data_settings['current_part']}/{data_settings['n_parts']}",
            'font_path': 'Arial Black',
            'font_size': 32,
            'x': video_configs['bg_size'][0] - 150,
            'y': 80 - 40,
            'spacing': 4,
            'align': 'center',
            'fill': '#000000',
            'max_line_length': 300,
        }
        draw.circle(
            [video_configs['bg_size'][0] - 115, 80, 300, 300],
            outline="#000000",
            width=4,
            radius=60,
            fill=(255, 255, 255, 200),
        )
        draw_resized_text_on_image(draw, part_text_settings, video_configs, is_centered=False)

    # 5. Words
    for i_row, row in df_filt.iterrows():
        # Chinese
        text_settings = {
            'text': row['chinese'],
            'font_path': video_configs['font_path'],
            'font_size': video_configs['words_settings']['font_size']['chinese'],
            'x': video_configs['words_settings']['x']['chinese'],
            'y': video_configs['words_settings']['y'] + i_row * (video_configs['words_settings']['font_size']['chinese'] + video_configs['words_settings']['spacing']),
            'spacing': video_configs['words_settings']['spacing'],
            'align': video_configs['words_settings']['align']['chinese'],
            'fill': video_configs['words_settings']['fill']['chinese'],
            'max_line_length': video_configs['words_settings']['max_line_length']['chinese'],
        }
        draw_resized_text_on_image(draw, text_settings, video_configs)

        # Pinyin
        text_settings = {
            'text': row['pinyin'],
            'font_path': video_configs['font_path'],
            'font_size': video_configs['words_settings']['font_size']['pinyin'],
            'x': video_configs['words_settings']['x']['pinyin'],
            'y': video_configs['words_settings']['y'] + i_row * (video_configs['words_settings']['font_size']['chinese'] + video_configs['words_settings']['spacing']),
            'spacing': video_configs['words_settings']['spacing'],
            'align': video_configs['words_settings']['align']['pinyin'],
            'fill': video_configs['words_settings']['fill']['pinyin'],
            'max_line_length': video_configs['words_settings']['max_line_length']['pinyin'],
        }
        draw_resized_text_on_image(draw, text_settings, video_configs)

        # English
        text_settings = {
            'text': row['english'],
            'font_path': video_configs['font_path'],
            'font_size': video_configs['words_settings']['font_size']['english'],
            'x': video_configs['words_settings']['x']['english'],
            'y': video_configs['words_settings']['y'] + i_row * (video_configs['words_settings']['font_size']['chinese'] + video_configs['words_settings']['spacing']),
            'spacing': video_configs['words_settings']['spacing'],
            'align': video_configs['words_settings']['align']['english'],
            'fill': video_configs['words_settings']['fill']['english'],
            'max_line_length': video_configs['words_settings']['max_line_length']['english'],
        }
        draw_resized_text_on_image(draw, text_settings, video_configs)

    no_hl_img_file_path = f"{data_settings['output_path_images']}/no_highlights.png"
    original_img.save(no_hl_img_file_path)
    return no_hl_img_file_path

# no_hl_img_file_path = draw_vocab_list_whole_image(video_configs, data_settings, df_filt[df_filt['chinese'].isin(vocab_word_list)])
# img_show = ImageClip(no_hl_img_file_path, duration=1).with_start(0)
# img_show.display_in_notebook()

# Generate video

In [ ]:
def create_video_without_highlights(data_settings, video_configs, no_hl_img_file_path):
    combined_audio = AudioFileClip(f"{data_settings['output_path_audio']}/!combined_part{data_settings['current_part']}.mp3")
    clips_no_highlights = [ImageClip(no_hl_img_file_path, duration=combined_audio.duration).with_start(0)]
    video = CompositeVideoClip(clips_no_highlights, size=video_configs['bg_size'])
    video.audio = combined_audio
    video.duration = combined_audio.duration
    no_highlights_video_path = f"{data_settings['output_path']}/{data_settings['shared_char']}_no_highlights_part{data_settings['current_part']}.mp4"
    video.write_videofile(no_highlights_video_path, fps=24)

create_video_without_highlights(data_settings, video_configs, no_hl_img_file_path)

In [ ]:
def create_video_with_highlights(df_durations, audio_settings, data_settings, video_configs):
    # Compute highlight durations
    highlight_start_ids = [1+x for x in df_durations[df_durations['audio_path']=='inter_word_pause'].index.tolist()]
    highlight_end_ids = [2+x for x in highlight_start_ids]
    dict_highlight_durations = defaultdict(list)
    for i_row, row in df_durations.iterrows():
        if i_row in highlight_start_ids:
            dict_highlight_durations['start_time'].append(row['start_time'] - audio_settings['pause_ms_within_word']/(2*1000))
        if i_row in highlight_end_ids:
            dict_highlight_durations['end_time'].append(row['end_time'] + audio_settings['pause_ms_within_word']/(2*1000))
    df_highlight_durations = pd.DataFrame(dict_highlight_durations)
    df_highlight_durations['duration'] = df_highlight_durations['end_time'] - df_highlight_durations['start_time']

    no_highlights_video_path = f"{data_settings['output_path']}/{data_settings['shared_char']}_no_highlights_part{data_settings['current_part']}.mp4"
    video = VideoFileClip(no_highlights_video_path)
    clips_with_highlights = [video]

    rect_width = video_configs['bg_size'][0] - 2*video_configs['words_settings']['max_line_length_buffer_size']['english'] + 2*video_configs['highlight_rect_x_buffer']
    rect_height = video_configs['words_settings']['font_size']['chinese'] + video_configs['words_settings']['spacing']
    rect_x = video_configs['words_settings']['max_line_length_buffer_size']['english'] - video_configs['highlight_rect_x_buffer']
    for i_row, row in df_highlight_durations.iterrows():
        rect_y = video_configs['words_settings']['y'] - \
            (video_configs['words_settings']['spacing']/2) + \
            i_row*(video_configs['words_settings']['spacing'] + video_configs['words_settings']['font_size']['chinese'])
        rect = (ColorClip(size=(rect_width, rect_height), color=video_configs['highlight_rect_color'], duration=row['duration'])
                .with_start(row['start_time'])
                .with_opacity(video_configs['highlight_rect_opacity'])
                .with_position((rect_x, rect_y)))
        clips_with_highlights.append(rect)

    # Overlay the rectangle on the video
    final_video = CompositeVideoClip(clips_with_highlights)
    final_video.write_videofile(f"{data_settings['output_path']}/{data_settings['shared_char']}_part{data_settings['current_part']}.mp4", codec="libx264")


create_video_with_highlights(df_durations, audio_settings, data_settings, video_configs)

# All in one

In [68]:
data_settings = {
    'shared_char': '车',
    'char_pinyin': 'chē',
    'char_english': 'vehicle',
    'max_priority': 4,
    'min_adu': 3,
    'min_per': 3,
    'types_allowed': WORD_TYPES + PROPER_NOUN_TYPES,
    'sort_cols': ['priority', 'cat_v3', 'pinyin'],
    'sort_ascending': [True, True, True],
    'words_rmv': ['马里奥赛车', '停车标志'],
    'n_words_per_video': 12,
    'current_part': 1,
}

In [78]:
print('Loading settings')
data_settings = enhance_data_settings(data_settings)
video_configs = load_video_configs()
create_directories(data_settings)

print('Loading and filtering data')
df_all_vocab = pd.read_csv('static/latest_data.csv')
df_filt = get_filtered_words(df_all_vocab, data_settings)
print(len(df_filt), [data_settings['shared_char']] + df_filt['chinese'].values.tolist())

print('Cut vocab into parts')
data_settings['n_words_total'] = len(df_filt)
data_settings['n_parts'] = int(np.ceil(len(df_filt) / data_settings['n_words_per_video']))

for current_part in range(1, data_settings['n_parts'] + 1):
    # Determine vocabulary in current part
    data_settings['current_part'] = current_part
    start_index = (current_part - 1) * data_settings['n_words_per_video']
    end_index = start_index + data_settings['n_words_per_video'] - 1
    data_settings['current_part_index_range'] = (start_index, end_index)
    print(f"Processing part {current_part}/{data_settings['n_parts']} with index range {data_settings['current_part_index_range']}")
    df_filt_currentpart = df_filt[
        (df_filt.index >= data_settings['current_part_index_range'][0]) &
        (df_filt.index <= data_settings['current_part_index_range'][1])
    ].reset_index(drop=True)

    print('Making audio')
    df_durations = stitch_audios(audio_settings, data_settings, df_filt_currentpart['chinese'].values.tolist())
    print('Making image')
    no_hl_img_file_path = draw_vocab_list_whole_image(video_configs, data_settings, df_filt_currentpart)
    print('Making video without highlights')
    create_video_without_highlights(data_settings, video_configs, no_hl_img_file_path)
    print('Making video with highlights')
    create_video_with_highlights(df_durations, audio_settings, data_settings, video_configs)

Loading settings
Loading and filtering data
23 ['车', '出租车', '电动车', '堵车', '公共汽车', '火车', '汽车', '自行车', '救护车', '晕车', '车库', '车位', '车祸', '自动驾驶汽车', '车牌', '共享单车', '卡车', '缆车', '摩托车', '停车场', '网约车', '购物车', '三轮车', '婴儿车']
Cut vocab into parts
Processing part 1/2 with index range (0, 11)
Making audio
Audio duration: 42.5s
Making image
Making video without highlights
MoviePy - Building video output/shared_char_shorts/车/车_no_highlights_part1.mp4.
MoviePy - Writing audio in 车_no_highlights_part1TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/车/车_no_highlights_part1.mp4



MoviePy - Done !
MoviePy - video ready output/shared_char_shorts/车/车_no_highlights_part1.mp4
Making video with highlights
MoviePy - Building video output/shared_char_shorts/车/车_part1.mp4.
MoviePy - Writing audio in 车_part1TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/车/车_part1.mp4



frame_index: 100%|█████████▉| 1017/1022 [00:21<00:00, 50.24it/s, now=None]/Users/scott/Library/Caches/pypoetry/virtualenvs/my-mandarin-database-XyoBA6Fv-py3.13/lib/python3.13/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file output/shared_char_shorts/车/车_no_highlights_part1.mp4, 2764800 bytes wanted but 0 bytes read at frame index 1021 (out of a total 1022 frames), at time 42.54/42.61 sec. Using the last valid frame instead.
  warnings.warn(


...

MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/车/车_no_highlights_part2.mp4



MoviePy - Done !
MoviePy - video ready output/shared_char_shorts/车/车_no_highlights_part2.mp4
Making video with highlights
MoviePy - Building video output/shared_char_shorts/车/车_part2.mp4.
MoviePy - Writing audio in 车_part2TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/车/车_part2.mp4



frame_index: 100%|█████████▉| 997/998 [00:20<00:00, 42.58it/s, now=None]/Users/scott/Library/Caches/pypoetry/virtualenvs/my-mandarin-database-XyoBA6Fv-py3.13/lib/python3.13/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file output/shared_char_shorts/车/车_no_highlights_part2.mp4, 2764800 bytes wanted but 0 bytes read at frame index 997 (out of a total 998 frames), at time 41.54/41.59 sec. Using the last valid frame instead.
  warnings.warn(


MoviePy - Done !
MoviePy - video ready output/shared_char_shorts/车/车_part2.mp4
